# BC loss as a fulfillment value → probabilistic handoff → BPG (walker + G1 standup)

**Pipeline.** A student `KnotPolicy` predicts the same object the expert optimizes — a
`(num_knots, nu)` control-knot trajectory — and trains online, DAgger-style, against a
**MPPIv2** expert using fullfillment terms. The behaviour-cloning loss is expressed as a
fulfillment value

$$f_{bc} = \exp\big(-\alpha\,\mathrm{MSE}(\pi_{knots}(s),\ \text{expert knots})\big) \in (0,1],\qquad \alpha = 2$$

and joins the task fulfillment terms inside the actor's FPL conjunction (Balanced Policy
Gradient).

$$\mathcal{L}_{actor} = -\mu_2\Big(u_{FPL}\big([\,FQ(s, \pi_0(s)),\ f_{bc}\,]\big)\Big)$$

**Probabilistic handoff.** The smoothed labeled-batch $f_{bc}$ *is the probability of
following the student*: at every env step, with probability $p_{student} = \bar f_{bc}$
the student's (exploration-noised) policy acts; otherwise the expert MPPIv2 **plans on
demand** and its action executes, yielding a fresh knot label ($\bar f_{bc}=0.9$ → 90%
student steps). The handoff is self-regulating — if the student drifts from the teacher
on labeled states, $\bar f_{bc}$ drops and the expert takes back control — so the
expensive MPPI planning fades out as $\bar f_{bc} \to 1$. In the actor loss the BC term
applies per-sample only where a label exists (unlabeled samples contribute the
conjunction identity 1), so BPG on the task terms carries the rest.

**Recorded metric.** Total cumulative env timesteps (expert- and student-acted alike)
until the **smoothed running combined task fulfillment over the student-acted steps**
(mean of `running_cost_terms_f`, 50-student-step moving window) first reaches **0.9** fullfillment.
Only student-acted steps feed the window; otherwise the expert's own high fulfillment
would satisfy the target while it still drives.

| hyperparameter | walker | g1_standup |
|---|---|---|
| expert K / noise / temp | 256 / 0.3 / 0.05 | 256 / 0.15 / 0.05 |
| plan horizon / knots / spline / iters | 0.6 s / 8 / cubic / 2 | 0.8 s / 6 / cubic / 2 |
| expert FPL | `use_fpl_cost`, p=0.1, **γ=0.99** | same |
| episode start | default pose | **scripted collapse** (fold 3 s + settle 2 s) |
| episode length / termination | 400 (torso z < 0.8) | 500 (none — starts fallen) |
| α (BC fulfillment) | 2.0 | same |
| p_student window (updates) | 50 | same |
| 0.9 target window (env steps) | 50 | same |
| BPG: γ / τ / α_FV / lr / batch | 0.9 / 0.005 / 0.75 / 1e-3 / 256 | same |
| grad updates per env step | 4 on expert steps, 1 on student steps | same |
| student exploration | parameter noise σ=0.1 | same |
| env-step budget | 500k | 500k |

Teacher γ **must** be 0.99: the FPL discount multiplies per-step fulfillment inside the
plan horizon, so γ=0.9 gives a ~10-step effective lookahead and a teacher that falls.

In [ ]:
import os, time, base64, tempfile
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import mujoco
import imageio
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

from analytic_mppi.tasks import make_task
from analytic_mppi.tasks.base import power_mean as np_power_mean
from analytic_mppi.dynamics import MujocoBackend
from analytic_mppi.controllers.mppi_v2 import MPPIv2

DEVICE = "cpu"
SEED   = 0
SMOKE  = os.environ.get("SMOKE", "0") == "1"   # SMOKE=1 -> minutes-scale self-test

# --- FPL / teacher ---
FPL_P             = 0.1    
TEACHER_FPL_GAMMA = 0.99   
GAMMA             = 0.9    

# --- BC-as-fulfillment + probabilistic handoff ---
ALPHA_BC   = 2.0    # f_bc = exp(-ALPHA_BC * knot MSE); 2.0 is what distills
FBC_WINDOW = 50     # window (in gradient updates) whose mean is p_student

# --- the recorded metric ---
TARGET_RUN_F = 0.9  # smoothed running combined fulfillment target
RUN_WINDOW   = 50   # moving window, in STUDENT-acted env steps only

# --- training ---
BATCH, LR, TAU, ALPHA_FV = 256, 1e-3, 0.005, 0.75
SIGMA            = 0.1       # parameter-space exploration noise (student steps)
EXPERT_UPD_PER_STEP  = 4     # grad updates on expert steps (a plan costs far
                             # more than the grad steps beside it)
STUDENT_UPD_PER_STEP = 1     # grad updates on student-acted steps
TOTAL_BUDGET     = 500_000   # cumulative env-step cap per env
BUFFER_CAP       = 500_000
RESET_NOISE      = 5e-3
EVAL_EPISODES    = 5

ENV_CFG = {
    "walker": dict(   # planar biped, must WALK at 1.5 m/s; root order z-then-x
        drop_qpos=(1,), healthy_z=0.8, init="zeros", terminate=True, horizon=400,
        teacher=dict(num_samples=256, noise_level=0.3, plan_horizon=0.6,
                     num_knots=8, spline_type="cubic", iterations=2)),
    "g1_standup": dict(  # 29-dof humanoid, standup from a scripted collapse
        drop_qpos=(0, 1), healthy_z=0.5, init="collapsed", terminate=False, horizon=500,
        teacher=dict(num_samples=256, noise_level=0.15, plan_horizon=0.8,
                     num_knots=6, spline_type="cubic", iterations=2)),
}

if SMOKE:
    TOTAL_BUDGET, FBC_WINDOW, RUN_WINDOW = 2_000, 20, 20
    EXPERT_UPD_PER_STEP, EVAL_EPISODES = 2, 2
    for _c in ENV_CFG.values():
        _c["teacher"].update(num_samples=64, iterations=1)
        _c["horizon"] = 100

torch.manual_seed(SEED)
np.random.seed(SEED)

RUN_DIR = "runs/bc_fulfillment_bpg"
os.makedirs(RUN_DIR, exist_ok=True)
_progress = open(os.path.join(RUN_DIR, "progress.log"), "a", buffering=1)

def log(*parts):
    """print + append to progress.log, for live monitoring."""
    msg = " ".join(str(p) for p in parts)
    print(msg)
    _progress.write(time.strftime("[%H:%M:%S] ") + msg + "\n")

log(f"SMOKE={SMOKE} | budget={TOTAL_BUDGET} | p_student = smoothed f_bc -> target={TARGET_RUN_F}")

In [ ]:
# === FPL utilities (torch) + networks ==========================================
def power_mean_t(x: torch.Tensor, p: float, dim: int = -1, eps: float = 1e-6):
    """Generalised power mean (mean_i x_i^p)^(1/p); clamped to [eps,1]."""
    x = x.clamp(eps, 1.0)
    if abs(p) < 1e-8:
        return torch.exp(x.log().mean(dim=dim))
    return x.pow(p).mean(dim=dim).pow(1.0 / p)


def u_fpl_flat(fq: torch.Tensor, p: float = FPL_P) -> torch.Tensor:
    """phi = AND^p over fulfillment terms — the teacher's per-step aggregation."""
    return power_mean_t(fq, p)


def mlp(sizes, act=nn.ReLU, out_act=nn.Identity):
    layers = []
    for i in range(len(sizes) - 1):
        layers += [nn.Linear(sizes[i], sizes[i + 1]),
                   act() if i < len(sizes) - 2 else out_act()]
    return nn.Sequential(*layers)


class KnotPolicy(nn.Module):
    """pi(s) -> (num_knots * nu) normalized control knots z in [-1,1] (tanh).

    The SAME object MPPIv2 optimizes (its mean knot trajectory). z[:nu] is the
    executed t=0 action: tk[0] = 0, so the spline value at t=0 equals knot 0 for
    every spline type."""
    def __init__(self, obs_dim, act_dim, n_knots, hidden=(256, 256)):
        super().__init__()
        self.act_dim, self.n_knots = act_dim, n_knots
        self.net = mlp([obs_dim, *hidden, n_knots * act_dim], out_act=nn.Tanh)

    def forward(self, s):
        return self.net(s)


class FQCritic(nn.Module):
    """Vector fulfillment Q-values FQ(s, a) in [0,1]^n_obj over the EXECUTED
    action (sigmoid keeps the FPL [0,1] semantics, matching y_TD)."""
    def __init__(self, obs_dim, act_dim, n_obj, hidden=(256, 256)):
        super().__init__()
        self.net = mlp([obs_dim + act_dim, *hidden, n_obj])

    def forward(self, s, a):
        return torch.sigmoid(self.net(torch.cat([s, a], dim=-1)))

In [ ]:
# === Replay buffer (+ FV^obs, + knot labels) + the mixture BPG agent ===========
def compute_fv_obs(R, gamma, truncated):
    """FV^obs_t = (1-g) sum_k g^k r_{t+k} (+ geometric tail if truncated)."""
    R = np.asarray(R, np.float32)
    S = np.zeros_like(R)
    nxt = np.zeros(R.shape[1], np.float32)
    for t in range(len(R) - 1, -1, -1):
        S[t] = R[t] + gamma * nxt
        nxt = S[t]
    fv = (1.0 - gamma) * S
    if truncated:
        rem = np.arange(len(R), 0, -1)
        fv += (gamma ** rem)[:, None] * R[-1][None, :]
    return fv.astype(np.float32)


class ReplayBuffer:
    """(s, a, r_vec, s2, term, FV^obs) + the expert's z-space knot label and a
    has_lab mask (1.0 on expert-acted transitions, 0.0 on student-acted)."""
    def __init__(self, cap, obs_dim, act_dim, n_obj, lab_dim):
        self.cap = cap
        self.s    = np.zeros((cap, obs_dim), np.float32)
        self.a    = np.zeros((cap, act_dim), np.float32)
        self.r    = np.zeros((cap, n_obj),   np.float32)
        self.s2   = np.zeros((cap, obs_dim), np.float32)
        self.term = np.zeros((cap, 1),       np.float32)
        self.fv   = np.zeros((cap, n_obj),   np.float32)
        self.lab  = np.zeros((cap, lab_dim), np.float32)
        self.has  = np.zeros((cap, 1),       np.float32)
        self.idx, self.full = 0, False

    def add_episode(self, S, A, R, S2, TERM, gamma, truncated,
                    labels=None, has_lab=None):
        FV = compute_fv_obs(R, gamma, truncated)
        for i in range(len(S)):
            j = self.idx
            self.s[j], self.a[j], self.r[j] = S[i], A[i], R[i]
            self.s2[j], self.term[j], self.fv[j] = S2[i], TERM[i], FV[i]
            self.lab[j] = labels[i]  if labels  is not None else 0.0
            self.has[j] = has_lab[i] if has_lab is not None else 0.0
            self.idx = (self.idx + 1) % self.cap
            self.full = self.full or self.idx == 0

    def sample(self, n):
        hi = self.cap if self.full else self.idx
        k = np.random.randint(0, hi, size=n)
        t = lambda x: torch.as_tensor(x[k], device=DEVICE)
        return (t(self.s), t(self.a), t(self.r), t(self.s2),
                t(self.term), t(self.fv), t(self.lab), t(self.has))

    def __len__(self):
        return self.cap if self.full else self.idx


class BCMixBPG:
    """BPG whose actor conjunction carries a BC fulfillment term.

    actor loss = -mu_2( u_fpl_flat([FQ(s, pi_0(s)), f_bc]) )
      labeled sample   : f_bc = exp(-ALPHA_BC * MSE(pi_knots(s), expert z-knots))
      unlabeled sample : f_bc = 1 (identity of the conjunction, no gradient)
    Returns the labeled-batch mean f_bc, which the driver uses as p_student.
    Critic step is stock BPG: mu_2 TD residual + ALPHA_FV * mu_2 FV^obs residual.
    """
    def __init__(self, obs_dim, act_dim, n_knots, n_obj, hidden=(256, 256)):
        self.obs_dim, self.act_dim = obs_dim, act_dim
        self.n_knots, self.n_obj, self.hidden = n_knots, n_obj, hidden
        self.actor    = KnotPolicy(obs_dim, act_dim, n_knots, hidden).to(DEVICE)
        self.actor_t  = KnotPolicy(obs_dim, act_dim, n_knots, hidden).to(DEVICE)
        self.critic   = FQCritic(obs_dim, act_dim, n_obj, hidden).to(DEVICE)
        self.critic_t = FQCritic(obs_dim, act_dim, n_obj, hidden).to(DEVICE)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.critic_t.load_state_dict(self.critic.state_dict())
        self.a_opt = torch.optim.Adam(self.actor.parameters(),  lr=LR)
        self.c_opt = torch.optim.Adam(self.critic.parameters(), lr=LR)

    @torch.no_grad()
    def act(self, s, actor=None):
        actor = actor if actor is not None else self.actor
        s = torch.as_tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        return actor(s).squeeze(0).cpu().numpy()          # (n_knots * act_dim,)

    def perturbed_actor(self, sigma):
        """Parameter-space noise: theta ~ N(theta, sigma|theta|)."""
        noisy = KnotPolicy(self.obs_dim, self.act_dim, self.n_knots,
                           self.hidden).to(DEVICE)
        noisy.load_state_dict(self.actor.state_dict())
        with torch.no_grad():
            for prm in noisy.parameters():
                prm.add_(sigma * prm.abs() * torch.randn_like(prm))
        return noisy

    def train_step(self, buf, batch_size):
        s, a, r, s2, term, fv, lab, has = buf.sample(batch_size)

        # --- critic: TD + observed-fulfillment regulariser (stock BPG) ---
        with torch.no_grad():
            a2 = self.actor_t(s2)[:, :self.act_dim]
            y = (1 - GAMMA) * r + GAMMA * (1 - term) * self.critic_t(s2, a2)
        fq = self.critic(s, a)
        l_td = torch.sqrt(((y  - fq) ** 2).mean())
        l_fv = torch.sqrt(((fv - fq) ** 2).mean())
        c_loss = l_td + ALPHA_FV * l_fv
        self.c_opt.zero_grad(); c_loss.backward(); self.c_opt.step()

        # --- actor: FPL conjunction of task FQ terms + the BC fulfillment ---
        k = self.actor(s)                                  # (B, n_knots*act_dim)
        fq_pi = self.critic(s, k[:, :self.act_dim])
        mse   = ((k - lab) ** 2).mean(dim=-1, keepdim=True)
        f_raw = torch.exp(-ALPHA_BC * mse)
        f_bc  = torch.where(has > 0, f_raw, torch.ones_like(f_raw))
        u = u_fpl_flat(torch.cat([fq_pi, f_bc], dim=-1))
        a_loss = -torch.sqrt((u ** 2).mean())
        self.a_opt.zero_grad(); a_loss.backward(); self.a_opt.step()

        # --- Polyak targets ---
        with torch.no_grad():
            for tp, sp in zip(self.actor_t.parameters(),  self.actor.parameters()):
                tp.mul_(1 - TAU).add_(TAU * sp)
            for tp, sp in zip(self.critic_t.parameters(), self.critic.parameters()):
                tp.mul_(1 - TAU).add_(TAU * sp)
        n_lab = float(has.sum())
        f_bc_lab = float((f_raw.detach() * has).sum() / n_lab) if n_lab > 0 else None
        return dict(l_td=float(l_td.detach()), l_fv=float(l_fv.detach()),
                    u=float(u.mean().detach()), f_bc_lab=f_bc_lab)

In [ ]:
# === G1 collapse: the scripted fold that produces the standup start state ======
def make_collapsed_state():
    """Fold the G1 to the floor (humanoid_standup_mppiv2.ipynb script).
    Returns (collapsed FULLPHYSICS state, every collapse state, for video)."""
    task = make_task("g1_standup")
    backend = MujocoBackend(task.model_path)
    fold = np.zeros(backend.nu)
    fold[[0, 6]]  = -1.8      # hip pitch
    fold[[3, 9]]  = 2.6       # knees
    fold[[4, 10]] = -0.6      # ankle pitch
    states = []
    n_fold, n_settle = round(3.0 / backend.dt), round(2.0 / backend.dt)
    for step in range(n_fold + n_settle):
        backend.step(fold if step < n_fold else np.zeros(backend.nu))
        states.append(backend.get_state().copy())
    torso_z = float(backend.data.sensordata[task._torso_pos_adr + 2])
    print(f"collapsed: pelvis z = {backend.data.qpos[2]:.3f}, torso z = {torso_z:.3f}")
    return backend.get_state().copy(), np.asarray(states)


STATE0_G1, COLLAPSE_STATES_G1 = make_collapsed_state()

In [ ]:
# === Environment wrapper (z-space actions) + fulfillment evaluation ============
class FPLEnv:
    """Repo task + MujocoBackend, vector fulfillment reward. 
    obs    = [qpos minus root-translation coords, qvel]
    action = z in [-1,1]^nu, mapped affinely onto actuator_ctrlrange
    reward = task.running_cost_terms_f in [0,1]^n_obj
    """

    def __init__(self, name, seed=SEED):
        cfg = ENV_CFG[name]
        self.name, self.cfg = name, cfg
        self.task = make_task(name)
        self.backend = MujocoBackend(self.task.model_path)
        self.keep_qpos = np.array(
            [i for i in range(self.backend.nq) if i not in cfg["drop_qpos"]])
        self.obs_dim = len(self.keep_qpos) + self.backend.nv
        self.act_dim = self.backend.nu
        self.horizon = cfg["horizon"]
        self.u_center = 0.5 * (self.task.u_max + self.task.u_min)
        self.u_half   = 0.5 * (self.task.u_max - self.task.u_min)
        self.rng = np.random.default_rng(seed)
        self.t = 0
        base = np.zeros(self.backend.nstate)
        if cfg["init"] == "collapsed":
            base = STATE0_G1.copy()
        elif cfg["init"].startswith("keyframe:"):
            kf = self.backend.model.keyframe(cfg["init"].split(":", 1)[1])
            base[self.backend.qpos_slice] = kf.qpos
        self._base_state = base
        self.reset()
        self.n_obj = int(self._r_probe().shape[-1])

    def to_z(self, u):
        """ctrl units -> normalized z (exact inverse of the step() mapping)."""
        return np.clip((u - self.u_center) / self.u_half, -1.0, 1.0)

    def _obs(self):
        d = self.backend.data
        return np.concatenate([np.asarray(d.qpos)[self.keep_qpos],
                               d.qvel]).astype(np.float32)

    def _r_probe(self):
        d = self.backend.data
        return self.task.running_cost_terms_f(
            d.qpos, d.qvel, d.sensordata, np.zeros(self.act_dim))

    def torso_z(self):
        return float(self.task._torso_height(self.backend.data.sensordata))

    def reset(self):
        state = self._base_state.copy()
        state[self.backend.qpos_slice] += self.rng.uniform(
            -RESET_NOISE, RESET_NOISE, self.backend.nq)
        state[self.backend.qvel_slice] += self.rng.uniform(
            -RESET_NOISE, RESET_NOISE, self.backend.nv)
        self.backend.set_state(state)               # set_state runs mj_forward
        self.t = 0
        return self._obs()

    def step(self, z):
        u = self.u_center + self.u_half * np.clip(
            np.asarray(z, dtype=np.float64), -1.0, 1.0)
        self.backend.step(u)
        # mj_step leaves sensordata at the PRE-step state; refresh before reading
        mujoco.mj_forward(self.backend.model, self.backend.data)
        d = self.backend.data
        r_vec = self.task.running_cost_terms_f(d.qpos, d.qvel, d.sensordata, u)
        self.t += 1
        terminated = self.cfg["terminate"] and self.torso_z() < self.cfg["healthy_z"]
        truncated = (not terminated) and (self.t >= self.horizon)
        return self._obs(), r_vec.astype(np.float32), terminated, truncated

    def full_state(self):
        return self.backend.get_state()


def episode_fulfillment(R, horizon):
    """sum_t power_mean(r_t, p) / horizon — a fallen episode scores 0 for every
    missing step (the G1 is passively stable; surviving-step averages mislead)."""
    return float(np.sum(np_power_mean(np.asarray(R), FPL_P, eps=1e-6)) / horizon)


def evaluate_fulfillment(agent, env_name, episodes=None, eval_seed=10_000):
    """Greedy rollouts on fresh, deterministically-seeded envs.
    Returns (mean episode fulfillment, per-term means)."""
    episodes = EVAL_EPISODES if episodes is None else episodes
    scores, term_means = [], []
    for e in range(episodes):
        env = FPLEnv(env_name, seed=eval_seed + e)
        s, R, done = env.reset(), [], False
        while not done:
            z = agent.act(s)[:env.act_dim]
            s, r_vec, term, trunc = env.step(z)
            R.append(r_vec)
            done = term or trunc
        scores.append(episode_fulfillment(R, env.horizon))
        term_means.append(np.mean(R, axis=0))
    return float(np.mean(scores)), np.mean(term_means, axis=0)


for _n in ENV_CFG:
    _e = FPLEnv(_n)
    print(f"{_n:11s} obs_dim={_e.obs_dim:3d} act_dim={_e.act_dim:3d} "
          f"n_obj={_e.n_obj} dt={_e.backend.dt}  reset torso z={_e.torso_z():.3f}")

In [ ]:
# === Expert: stock MPPIv2 (FPL mode) + pre-shift knot-label capture ============
def make_teacher(env, seed=SEED):
    return MPPIv2(env.task, env.backend, temperature=0.05, seed=seed,
                  use_fpl_cost=True, fpl_p=FPL_P, fpl_gamma=TEACHER_FPL_GAMMA,
                  **ENV_CFG[env.name]["teacher"])


_label_checked = set()

def expert_plan_and_act(ctrl, state, gap_steps=1):
    """Replicates MPPIv2.act() so the BC label is the plan for the CURRENT step.

    act() warm-start-shifts self.mean by dt BEFORE returning (sampling_base.py),
    so ctrl.mean read after act() is the NEXT step's warm start — the label must
    be copied between update_mean and _shift_mean. The controller stays stock.

    gap_steps: env steps since the expert last planned (the student drove the
    gap). The warm-start mean is shifted by the extra (gap_steps-1)*dt so the
    stale plan aligns with the present before resampling around it."""
    if gap_steps > 1:
        ctrl._shift_mean((gap_steps - 1) * ctrl.backend.dt)
    for _ in range(ctrl.iterations):
        traj = ctrl._rollout_and_score(state)
        ctrl.mean = ctrl.update_mean(traj)
        ctrl.last_trajectory = traj
    label = ctrl.mean.copy()                  # (num_knots, nu), the current plan
    u0 = ctrl.mean[0].copy()
    ctrl._shift_mean(ctrl.backend.dt)
    if id(ctrl) not in _label_checked:
        assert label.shape == (ctrl.num_knots, ctrl.nu), label.shape
        assert np.array_equal(u0, label[0]), "u0 must equal knot 0"
        _label_checked.add(id(ctrl))
    return u0, label

In [ ]:
# === Driver: per-step Bernoulli(p_student) mixture of student and expert ======
def train_env(env_name, log=log):
    """One continuous loop. Each env step:
      p_student = mean of the last FBC_WINDOW labeled-batch f_bc values
      coin ~ Bernoulli(p_student):  heads -> student (noisy actor) acts
                                    tails -> expert MPPIv2 plans on demand + labels
    Expert steps get EXPERT_UPD_PER_STEP grad updates, student steps
    STUDENT_UPD_PER_STEP. Stops at the first RUN_WINDOW-smoothed running
    fulfillment >= TARGET_RUN_F, or at TOTAL_BUDGET."""
    cfg = ENV_CFG[env_name]
    env = FPLEnv(env_name, seed=SEED)
    teacher = make_teacher(env)
    n_knots = cfg["teacher"]["num_knots"]
    agent = BCMixBPG(env.obs_dim, env.act_dim, n_knots, env.n_obj)
    buf = ReplayBuffer(BUFFER_CAP, env.obs_dim, env.act_dim, env.n_obj,
                       n_knots * env.act_dim)
    coin = np.random.default_rng(SEED + 7)
    fbc_win, run_win = deque(maxlen=FBC_WINDOW), deque(maxlen=RUN_WINDOW)
    hist = dict(f_bc=[], p_student=[], expert_flag=[], run_f=[], run_step=[],
                teacher_f=[], expert_queries=0, steps_to_p09=None,
                steps_to_target=None, expert_states=None, wall=0.0)
    total, ep = 0, 0
    t0 = time.perf_counter()

    while total < TOTAL_BUDGET and hist["steps_to_target"] is None:
        s = env.reset()
        teacher.reset()
        noisy = agent.perturbed_actor(SIGMA)          # exploration, student steps
        S, A, R, S2, TERM, LAB, HAS = [], [], [], [], [], [], []
        states = [env.full_state().copy()]
        done = truncated = False
        gap = 0                                       # steps since last expert plan
        while not done:
            p_student = float(np.mean(fbc_win)) if len(fbc_win) == FBC_WINDOW else 0.0
            if coin.uniform() < p_student:            # --- student acts ---
                z0 = agent.act(s, actor=noisy)[:env.act_dim]
                z_lab, has = np.zeros(n_knots * env.act_dim, np.float32), 0.0
                gap += 1
            else:                                     # --- expert acts + labels ---
                u0, knots_u = expert_plan_and_act(teacher, env.full_state(),
                                                  gap_steps=gap + 1)
                z0 = env.to_z(u0)
                z_lab, has = env.to_z(knots_u).ravel().astype(np.float32), 1.0
                gap = 0
                hist["expert_queries"] += 1
            s2, r_vec, term, truncated = env.step(z0)
            S.append(s); A.append(np.asarray(z0, np.float32)); R.append(r_vec)
            S2.append(s2); TERM.append([float(term)])
            LAB.append(z_lab); HAS.append([has])
            states.append(env.full_state().copy())
            s = s2
            total += 1
            hist["p_student"].append(p_student)
            hist["expert_flag"].append(has)
            if not has:               # only student-acted steps feed the metric
                comb = float(r_vec.mean())
                run_win.append(comb)
                hist["run_f"].append(comb); hist["run_step"].append(total)
                if len(run_win) == RUN_WINDOW and np.mean(run_win) >= TARGET_RUN_F:
                    hist["steps_to_target"] = total
                    log(f"[{env_name}] >>> smoothed running fulfillment over "
                        f"student steps reached {TARGET_RUN_F} at total step {total}")
            done = term or truncated or hist["steps_to_target"] is not None
            # --- gradient updates (on past, completed episodes) ---
            n_upd = EXPERT_UPD_PER_STEP if has else STUDENT_UPD_PER_STEP
            for _ in range(n_upd):
                if len(buf) < BATCH:
                    break
                info = agent.train_step(buf, BATCH)
                if info["f_bc_lab"] is not None:
                    hist["f_bc"].append(info["f_bc_lab"])
                    fbc_win.append(info["f_bc_lab"])
                    if (hist["steps_to_p09"] is None and len(fbc_win) == FBC_WINDOW
                            and np.mean(fbc_win) >= 0.9):
                        hist["steps_to_p09"] = total
                        log(f"[{env_name}] p_student reached 0.9 at total step {total}")
        if S:
            buf.add_episode(np.array(S, np.float32), np.array(A, np.float32),
                            np.array(R, np.float32), np.array(S2, np.float32),
                            np.array(TERM, np.float32), GAMMA,
                            truncated or hist["steps_to_target"] is not None,
                            labels=np.array(LAB, np.float32),
                            has_lab=np.array(HAS, np.float32))
        ep += 1
        n_exp = int(sum(h[0] for h in HAS))
        if n_exp == len(S):                           # pure-expert episode
            hist["teacher_f"].append(episode_fulfillment(R, env.horizon))
            if hist["expert_states"] is None:         # keep one for the video
                hist["expert_states"] = np.asarray(states)
        if ep <= 3 or ep % 25 == 0:
            log(f"[{env_name}] ep {ep:4d}  total={total:7d}  "
                f"p_student={p_student:.3f}  expert {n_exp}/{len(S)} steps  "
                f"run_f(win)={np.mean(run_win) if run_win else 0:.3f}")
    hist["wall"] = time.perf_counter() - t0

    q, frac = hist["expert_queries"], hist["expert_queries"] / max(total, 1)
    log(f"[{env_name}] done: steps_to_{TARGET_RUN_F}={hist['steps_to_target']}, "
        f"p09@{hist['steps_to_p09']}, expert queries {q} ({100*frac:.1f}% of "
        f"{total} steps), wall={hist['wall']:.0f}s")
    return agent, env, hist

In [ ]:
# === Plot + render helpers =====================================================
INK, MUTED, BLUE, ORANGE = "#383835", "#898781", "#2a78d6", "#d97706"


def plot_history(hist, env_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    ax = axes[0]
    if hist["f_bc"]:
        f = np.asarray(hist["f_bc"])
        w = min(FBC_WINDOW, len(f))
        sm = np.convolve(f, np.ones(w) / w, mode="valid")
        ax.plot(f, color=BLUE, alpha=0.25, lw=0.8)
        ax.plot(np.arange(w - 1, len(f)), sm, color=BLUE, lw=1.8,
                label=f"labeled-batch f_bc ({w}-update mean = p_student)")
    ax.axhline(0.9, color=MUTED, ls="--", lw=1)
    ax.text(0.02, 0.92, "p_student = 0.9", color=MUTED,
            fontsize=8, transform=ax.get_yaxis_transform())
    ax.set(xlabel="gradient update (labeled batches)", ylabel="BC fulfillment",
           title=f"{env_name} — f_bc, the student's handoff probability",
           ylim=(0, 1.02))
    ax.legend(loc="lower right", fontsize=8, frameon=False)

    ax = axes[1]
    if hist["p_student"]:
        x_all = np.arange(1, len(hist["p_student"]) + 1)
        ax.plot(x_all, np.asarray(hist["p_student"]), color=BLUE, lw=1.2,
                alpha=0.8, label="p_student")
        ef = np.asarray(hist["expert_flag"], np.float32)
        we = min(500, len(ef))
        ax.plot(x_all[we - 1:], np.convolve(ef, np.ones(we) / we, mode="valid"),
                color=MUTED, lw=1.0, ls="-.", label=f"expert fraction ({we}-step)")
    if hist["run_f"]:
        f = np.asarray(hist["run_f"]); x = np.asarray(hist["run_step"])
        w = min(RUN_WINDOW, len(f))
        sm = np.convolve(f, np.ones(w) / w, mode="valid")
        ax.plot(x, f, color=ORANGE, alpha=0.15, lw=0.5)
        ax.plot(x[w - 1:], sm, color=ORANGE, lw=1.8,
                label=f"student-step fulfillment ({w}-step mean)")
    ax.axhline(TARGET_RUN_F, color=MUTED, ls="--", lw=1)
    if hist["steps_to_target"] is not None:
        ax.axvline(hist["steps_to_target"], color=INK, ls=":", lw=1)
        ax.text(hist["steps_to_target"], 0.95,
                f" {TARGET_RUN_F} @ {hist['steps_to_target']}",
                color=INK, fontsize=8)
    ax.set(xlabel="cumulative env step", ylabel="fulfillment / probability",
           title=f"{env_name} — mixture rollout (metric: student steps only)",
           ylim=(0, 1.02))
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    for a in axes:
        for side in ("top", "right"):
            a.spines[side].set_visible(False)
    plt.tight_layout(); plt.show()


def greedy_rollout_states(agent, env_name, seed=20_000):
    env = FPLEnv(env_name, seed=seed)
    s, done = env.reset(), False
    states, R = [env.full_state().copy()], []
    while not done:
        s, r_vec, term, trunc = env.step(agent.act(s)[:env.act_dim])
        states.append(env.full_state().copy()); R.append(r_vec)
        done = term or trunc
    return np.asarray(states), np.asarray(R)


def render_html(model, states, caption, *, width=480, height=320, stride=None,
                min_seconds=None, track_qpos_idx=None, lookat=None,
                distance=3.0, elevation=-15.0, azimuth=135.0):
    """states (T, nstate FULLPHYSICS) -> self-contained <video> HTML."""
    min_seconds = (5.0 if SMOKE else 30.0) if min_seconds is None else min_seconds
    states = np.asarray(states)
    qpos = states[:, 1:1 + model.nq]
    qvel = states[:, 1 + model.nq:1 + model.nq + model.nv]
    if stride is None:
        stride = max(1, len(qpos) // 900)
    data = mujoco.MjData(model)
    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(model, cam)
    cam.distance, cam.elevation, cam.azimuth = distance, elevation, azimuth
    if lookat is not None:
        cam.lookat[:] = lookat
    r = mujoco.Renderer(model, height=height, width=width)
    frames = []
    for qp, qv in zip(qpos[::stride], qvel[::stride]):
        data.qpos[:], data.qvel[:] = qp, qv
        mujoco.mj_forward(model, data)
        if track_qpos_idx is not None:
            cam.lookat[0] = qp[track_qpos_idx]
        r.update_scene(data, cam)
        frames.append(r.render())
    r.close()
    fps = min(60.0, max(1.0, len(frames) / float(min_seconds)))
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        path = f.name
    imageio.mimwrite(path, np.asarray(frames), fps=fps, codec="libx264")
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    os.remove(path)
    return (f'<figure style="display:inline-block;text-align:center">'
            f'<figcaption><b>{caption}</b> &middot; {len(frames) / fps:.0f}s'
            f'</figcaption><video controls autoplay loop muted width="{width}">'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
            f'</video></figure>')


RESULTS = {}

## Walker

Planar biped, walk at 1.5 m/s. The run starts all-expert ($p_{student}=0$); as the
student clones the 8-knot plans, $p_{student}=\bar f_{bc}$ rises and the student takes
over a matching share of steps, until the 50-step smoothed running fulfillment of the
mixture reaches 0.9.

In [ ]:
t_wall = time.perf_counter()
agent_w, env_w, hist_w = train_env("walker")
eval_w, terms_w = evaluate_fulfillment(agent_w, "walker")
hist_w["wall_total"] = time.perf_counter() - t_wall
hist_w["eval"], hist_w["eval_terms"] = eval_w, terms_w
RESULTS["walker"] = hist_w
print(f"\nwalker greedy eval: {eval_w:.3f}  per-term "
      f"{dict(zip(env_w.task.cost_term_names_f, np.round(terms_w, 3)))}")
plot_history(hist_w, "walker")

In [ ]:
student_states_w, _ = greedy_rollout_states(agent_w, "walker")
html_w = (render_html(env_w.backend.model, hist_w["expert_states"],
                      "walker — expert MPPIv2 (FPL)", track_qpos_idx=1,
                      lookat=[0, 0, 0.8], distance=4.0, elevation=-10.0, azimuth=90.0)
          + render_html(env_w.backend.model, student_states_w,
                        "walker — student (mixture-trained)", track_qpos_idx=1,
                        lookat=[0, 0, 0.8], distance=4.0, elevation=-10.0, azimuth=90.0))
display(HTML(html_w))

## G1 standup

29-dof humanoid starting from the scripted collapse (videos prepend the fold, so the
clips show collapse → standup). No early termination — the robot starts fallen, and the
fulfillment terms (orientation / height / nominal) carry the signal. Same probabilistic
handoff.

In [ ]:
t_wall = time.perf_counter()
agent_g, env_g, hist_g = train_env("g1_standup")
eval_g, terms_g = evaluate_fulfillment(agent_g, "g1_standup")
hist_g["wall_total"] = time.perf_counter() - t_wall
hist_g["eval"], hist_g["eval_terms"] = eval_g, terms_g
RESULTS["g1_standup"] = hist_g
print(f"\ng1_standup greedy eval: {eval_g:.3f}  per-term "
      f"{dict(zip(env_g.task.cost_term_names_f, np.round(terms_g, 3)))}")
plot_history(hist_g, "g1_standup")

In [ ]:
student_states_g, _ = greedy_rollout_states(agent_g, "g1_standup")
expert_g = np.concatenate([COLLAPSE_STATES_G1, hist_g["expert_states"]])
student_g = np.concatenate([COLLAPSE_STATES_G1, student_states_g])
html_g = (render_html(env_g.backend.model, expert_g,
                      "G1 standup — expert MPPIv2 (FPL)")
          + render_html(env_g.backend.model, student_g,
                        "G1 standup — student (mixture-trained)"))
display(HTML(html_g))

In [ ]:
# === Results ===================================================================
rows = ["| env | teacher f | expert queries | steps to p_student≥0.9 | "
        "**steps to %.1f smoothed** | greedy eval | per-term | wall |"
        % TARGET_RUN_F,
        "|---|---|---|---|---|---|---|---|"]
for name, h in RESULTS.items():
    tf = f"{np.mean(h['teacher_f']):.3f}" if h["teacher_f"] else "—"
    tot = len(h["p_student"])
    q = f"{h['expert_queries']} ({100 * h['expert_queries'] / max(tot, 1):.0f}%)"
    p09 = h["steps_to_p09"] if h["steps_to_p09"] is not None else "—"
    reach = (f"**{h['steps_to_target']}**" if h["steps_to_target"] is not None
             else f"not reached ≤ {TOTAL_BUDGET}")
    terms = np.round(h["eval_terms"], 2).tolist()
    rows.append(f"| {name} | {tf} | {q} | {p09} | {reach} | {h['eval']:.3f} | "
                f"{terms} | {h['wall']:.0f}s |")
display(Markdown("\n".join(rows)))

for name, h in RESULTS.items():
    if h["steps_to_target"] is None:
        print(f"{name}: 0.9 smoothed running fulfillment NOT reached within "
              f"{TOTAL_BUDGET} env steps (budget cap).")

### Notes

- **The handoff is a per-step Bernoulli draw**: `p_student` is the 50-update moving mean
  of the labeled-batch BC fulfillment, so f_bc = 0.9 means a 90% chance the
  student's action executes at that step. The expert plans **only** when 
  it's picked (its warm-start mean time-shifted across the skipped steps), so MPPI cost decays as
  the student earns control — and returns automatically if the student drifts and f_bc
  drops. This is DAgger's mixture policy with a *learned, self-measured* β.
- **The BC term never switches off**: it stays in the actor conjunction for the whole
  run, but applies per-sample only where an expert label exists; unlabeled (student)
  samples contribute the conjunction identity 1. The rest of the objective is plain BPG
  on the same task fulfillment terms the teacher plans with.
- **Steps-to-0.9** counts all env steps, expert- and student-acted alike, since both
  consume simulator time — but the 0.9 window is fed **only by student-acted steps**: an
  early all-expert stretch scoring 0.9 is the teacher's competence, not the student's
  (the first run of this notebook "crossed" at step 323 that way, 100% expert-driven).
- Single seed (0); treat modest numbers as indicative, not significant.